In [4]:
from unsloth import FastVisionModel
from datasets import load_dataset
from dotenv import load_dotenv
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

load_dotenv()

True

In [ ]:
# TODO: update with front/back details
field_structure = {
    "full_name": "string (arabic)",
    "national_id": "string, 14 digits",
    "address": "string (arabic)",
}

In [ ]:
# TODO: update to load images and fields separately
dataset = load_dataset("/path/to/dataset", split="train")

In [ ]:
SYSTEM_PROMPT = f'''
    You are a Vision Language Model tasked with extracted field values from an Egyptian national identity
    document. You must extract the fields without making any changes to the fields and return them
    as they are in Arabic script.
'''
USER_PROMPT = f'''
    Extract all the fields out of this Egyptian national ID following this format: {field_structure}.
    Return all fields as they appear and do not make any changes or updates to any of the fields.
'''

In [ ]:
def generate_conversation(data):
    image = data['image']
    full_name = data['full_name']
    national_id = data['national_id']
    address = data['address']
    message = f'full name: {full_name}, national_id: {national_id}, address: f{address}'
    conversation = [
        {
            'role': 'system',
            'content': [
                    {
                        'type': 'text',
                        'text': SYSTEM_PROMPT
                    }
            ]
        },
        {
            'role': 'user', 
            'content': [
                {
                    'type': 'text', 'text': USER_PROMPT 
                },
                {
                    'type': 'image', 
                    'image': image
                }
            ]
        },
        {
            'role': 'assistant', 
            'content': [
                {
                    'type': 'text', 
                    'text': message
                }
            ]
        }
    ]
    return conversation

In [ ]:
training_data = []
for sample in dataset:
    training_data.append(generate_conversation(sample))


In [ ]:
# model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-8B-Instruct",
#                                                    load_in_4bit = False,
#                                                    use_gradient_checkpointing=True)


model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3.5-0.8B",
                                                   load_in_4bit = False,
                                                   use_gradient_checkpointing=True)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)

In [ ]:
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_data,
    args=SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        learning_rate = 2e-4, #TODO tune these hyperparameters?
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        output_dir = "models",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048)
)


NameError: name 'model' is not defined

#### Train the model

In [ ]:

trainer_stats = trainer.train()

#### Save the model and tokenizer

In [ ]:
model.save_pretrained("qwen_vlm")
tokenizer.save_pretrained("qwen_vlm")